# Calibration, clinical utility and SHAP: v5 models (main analysis)

Self-contained: reads files already produced by the pipeline; does not query BigQuery.

| Part | Question | Source |
|---|---|---|
| 1. Calibration | Do predicted probabilities match observed outcomes? | `predicciones_oof.csv` (Platt-calibrated probability) |
| 2. Clinical utility | Does the model give more net benefit than treating all / none? | same |
| 3. Out-of-fold SHAP | Which variables drive each model? How much weight does HRV get, and of what type? | per-fold refit with the stored hyperparameters |

**Scope fixed before looking at these results**
- Feature sets: the two of H1/H3, `clinico_sofa` and `integrada_sofa`. All three outcomes. Main analysis.
- Calibration and utility: all model families present in the predictions.
- SHAP: **Elastic Net** (exact, `LinearExplainer`) and **XGBoost** (exact, `TreeExplainer`), both on the log-odds scale. No "best model" is chosen for explanation.
- Calibration is judged with the hierarchy of Van Calster et al. (2019): mean (intercept ~ 0), weak (slope ~ 1) and moderate (curve).

Run time: 10-20 min on CPU.

## 0. Setup

In [ ]:
!pip install -q shap
from google.colab import drive
drive.mount('/content/drive')

import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from xgboost import XGBClassifier
import shap
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

RUTA_BASE = '/content/drive/MyDrive/tesis_sepsis'
RUTA_DATASET = f'{RUTA_BASE}/ultimate_dataset_v3_1.csv'
RUTA_COLUMNAS = f'{RUTA_BASE}/columnas_v3_1.json'
RUTA_MODELADO = f'{RUTA_BASE}/modelado_v5_1'
SALIDA = f'{RUTA_MODELADO}/calibracion_shap'
os.makedirs(SALIDA, exist_ok=True)

SEMILLA, N_EXTERNAS, N_REPLICAS_CAL = 42, 5, 1000
CONJUNTOS_H = ['clinico_sofa', 'integrada_sofa']
DESENLACES = {'sofa_respuesta': ('label_sofa_responder', 'Respuesta SOFA'),
              'hemodinamico':   ('label_cardio', 'Respuesta hemodinámica'),
              'progresion':     ('label_sofa_deterioro_2pts', 'Progresión SOFA ≥2')}
COLOR = {'clinico_sofa': '#1f77b4', 'integrada_sofa': '#d62728'}
print('shap', shap.__version__)

## 1. Data, feature sets and out-of-fold predictions

In [ ]:
df = pd.read_csv(RUTA_DATASET)
cols = json.load(open(RUTA_COLUMNAS))
# same transformations as pipeline_modelado_v5.cargar_datos()
df['sexo_masculino'] = df['gender'].astype(str).str.upper().str.startswith('M').astype(int)
for c in ['map_t0', 'mbp_mean_actual12h']:
    df.loc[(df[c] < 20) | (df[c] > 200), c] = np.nan
if 'lactato_medido_12h' not in df.columns:
    df['lactato_medido_12h'] = df['lactate_last_actual12h'].notna().astype(int)

CLIN = [c if c != 'gender' else 'sexo_masculino' for c in cols['features_clinicas']]
PAMLAC = list(dict.fromkeys(cols['features_pam_lactato_actual'] + ['lactato_medido_12h']))
SOFA = cols['features_sofa']
HRV = cols['features_hrv']
CONJ = {'clinico_sofa': CLIN + PAMLAC + SOFA, 'integrada_sofa': HRV + CLIN + PAMLAC + SOFA}

# Groups for aggregated SHAP
DOMINIO = {'SDNN': 'HRV temporal', 'RMSSD': 'HRV temporal', 'SD1': 'HRV temporal', 'SD2': 'HRV temporal',
           'LF': 'HRV frecuencial', 'HF': 'HRV frecuencial', 'LFHF': 'HRV frecuencial',
           'SampEn': 'HRV no lineal', 'ApEn': 'HRV no lineal', 'DFA_a1': 'HRV no lineal'}
def grupo(v):
    if v in SOFA: return 'SOFA'
    if v in PAMLAC: return 'PAM y lactato (12 h)'
    if v in CLIN: return 'Clínicas en T0'
    if v == 'horas_desde_ultima_hrv': return 'HRV temporal'
    return DOMINIO[v.rsplit('_', 1)[0]]
def resumen(v):
    if v in HRV and v != 'horas_desde_ultima_hrv':
        s = v.rsplit('_', 1)[1]
        return {'mean': 'Nivel (media)', 'last': 'Nivel (último)', 'sd': 'Variabilidad (DE)', 'slope': 'Tendencia (pendiente)'}[s]
    return None

pred = pd.read_csv(f'{RUTA_MODELADO}/predicciones_oof.csv')
pred = pred[(pred.analisis == 'principal') & pred.conjunto.isin(CONJUNTOS_H)]
reg = pd.read_csv(f'{RUTA_MODELADO}/registro_configuraciones.csv')
reg = reg[(reg.analisis == 'principal') & reg.conjunto.isin(CONJUNTOS_H)]
print('Predicciones OOF:', pred.groupby(['desenlace', 'conjunto']).familia.nunique().to_dict())
print('Variables: clinico_sofa', len(CONJ['clinico_sofa']), '| integrada_sofa', len(CONJ['integrada_sofa']))

## 2. Calibration metrics
- **Calibration intercept** (mean calibration): `logit P(y=1) = a + logit(p)`. Ideal 0; > 0 = risk underestimated, < 0 = overestimated.
- **Calibration slope** (weak calibration): `logit P(y=1) = a + b*logit(p)`. Ideal 1; < 1 = predictions too extreme (overfitting), > 1 = too moderate.
- **E/O**: expected / observed events. Ideal 1.
- **Scaled Brier** = 1 - Brier / Brier of the null model (predicting the prevalence). > 0 = better than no information.
- **ICI** (integrated calibration index, Austin & Steyerberg 2019): mean distance between the LOWESS curve and the diagonal.
- 95 % CIs from a **patient-level bootstrap** (1000 replicates).

In [ ]:
def _logit(p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

def metricas_cal(y, p):
    lp = _logit(p)
    try:
        a = sm.GLM(y, np.ones((len(y), 1)), family=sm.families.Binomial(), offset=lp).fit().params[0]
        b = sm.GLM(y, sm.add_constant(lp), family=sm.families.Binomial()).fit().params[1]
    except Exception:
        a, b = np.nan, np.nan
    br = brier_score_loss(y, p); prev = y.mean()
    suav = lowess(y, p, frac=0.75, it=0, return_sorted=False)
    return {'intercepto': a, 'pendiente': b, 'E/O': p.sum() / max(y.sum(), 1),
            'brier': br, 'brier_escalado': 1 - br / (prev * (1 - prev)), 'ICI': np.mean(np.abs(suav - p))}

rng = np.random.default_rng(2026)
filas = []
for (des, cj, fam), g in pred.groupby(['desenlace', 'conjunto', 'familia']):
    y = g.y.to_numpy(); p = g.p_calibrada.to_numpy(); pid = g.icustay_id.to_numpy()
    punto = metricas_cal(y, p)
    u = np.unique(pid); idx_de = {k: np.flatnonzero(pid == k) for k in u}
    boot = []
    for _ in range(N_REPLICAS_CAL):
        i = np.concatenate([idx_de[k] for k in rng.choice(u, len(u))])
        if 0 < y[i].sum() < len(i): boot.append(metricas_cal(y[i], p[i]))
    boot = pd.DataFrame(boot)
    fila = {'desenlace': des, 'conjunto': cj, 'familia': fam, 'n': len(y), 'eventos': int(y.sum()),
            'auroc': roc_auc_score(y, g.p_sin_calibrar), 'auprc': average_precision_score(y, g.p_sin_calibrar)}
    for k, v in punto.items():
        fila[k] = v
        fila[f'{k}_ic'] = f'{boot[k].quantile(.025):.3f}–{boot[k].quantile(.975):.3f}'
    filas.append(fila)
    print(f'{des:15s} {cj:15s} {fam:14s} listo')

cal = pd.DataFrame(filas)
cal.to_csv(f'{SALIDA}/calibracion_metricas.csv', index=False)
vista = cal[['desenlace', 'conjunto', 'familia', 'auroc', 'intercepto', 'intercepto_ic', 'pendiente', 'pendiente_ic',
             'E/O', 'brier_escalado', 'ICI']].round(3)
display(vista)

## 3. Calibration curves
Points = deciles of predicted probability (quintiles for progression, given the few events) with Wilson 95 % CIs; line = LOWESS; bottom = histogram of predictions. One row per family, one column per outcome; blue = clinical + SOFA, red = integrated + SOFA.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return np.nan, np.nan
    p = k / n; d = 1 + z**2 / n; c = (p + z**2 / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return c - h, c + h

familias = sorted(pred.familia.unique())
fig, ax = plt.subplots(len(familias), 3, figsize=(15, 4.2 * len(familias)), squeeze=False)
for j, (des, (_, titulo)) in enumerate(DESENLACES.items()):
    for i, fam in enumerate(familias):
        a = ax[i, j]; lim = 0
        for cj in CONJUNTOS_H:
            g = pred[(pred.desenlace == des) & (pred.conjunto == cj) & (pred.familia == fam)]
            if g.empty: continue
            y, p = g.y.to_numpy(), g.p_calibrada.to_numpy()
            nb = 5 if des == 'progresion' else 10
            q = pd.qcut(p, nb, duplicates='drop')
            t = pd.DataFrame({'y': y, 'p': p, 'q': q}).groupby('q', observed=True).agg(pm=('p', 'mean'), k=('y', 'sum'), n=('y', 'size'))
            ci = np.array([wilson(k, n) for k, n in zip(t.k, t.n)])
            obs = t.k / t.n
            a.errorbar(t.pm, obs, yerr=[obs - ci[:, 0], ci[:, 1] - obs], fmt='o', ms=4, color=COLOR[cj], alpha=.8, capsize=2)
            s = lowess(y, p, frac=.75, it=0)
            a.plot(s[:, 0], s[:, 1], color=COLOR[cj], lw=2,
                   label=f"{cj} (int {cal.query('desenlace==@des & conjunto==@cj & familia==@fam').intercepto.iloc[0]:+.2f}, "
                         f"pend {cal.query('desenlace==@des & conjunto==@cj & familia==@fam').pendiente.iloc[0]:.2f})")
            lim = max(lim, np.nanmax(ci[:, 1]), p.max())
        lim = min(1, lim * 1.05)
        a.plot([0, lim], [0, lim], 'k--', lw=1); a.set_xlim(0, lim); a.set_ylim(0, lim)
        a.set_title(f'{titulo} · {fam}', fontsize=10); a.legend(fontsize=7, loc='upper left')
        if i == len(familias) - 1: a.set_xlabel('Probabilidad predicha (calibrada)')
        if j == 0: a.set_ylabel('Proporción observada')
        ins = a.inset_axes([0.55, 0.04, 0.42, 0.18])
        for cj in CONJUNTOS_H:
            g = pred[(pred.desenlace == des) & (pred.conjunto == cj) & (pred.familia == fam)]
            ins.hist(g.p_calibrada, bins=30, range=(0, lim), color=COLOR[cj], alpha=.5)
        ins.set_xticks([]); ins.set_yticks([])
plt.tight_layout(); plt.savefig(f'{SALIDA}/curvas_calibracion.png', dpi=200, bbox_inches='tight'); plt.show()

## 4. Clinical utility: decision curve analysis (Vickers & Elkin 2006)
**Net benefit** = TP/n - FP/n x (pt / (1 - pt)), where pt is the threshold probability for acting.

The "event" is what would prompt action (escalating treatment or closer monitoring):
- SOFA improvement and haemodynamic stability or recovery: event = **no** improvement / **no** stability (probability = 1 - p).
- Progression: event = progression.

A model is useful at thresholds where its curve lies above both "treat all" and "treat none".

In [ ]:
def beneficio_neto(y, p, umbrales):
    n = len(y); out = []
    for t in umbrales:
        pos = p >= t; vp = np.sum(pos & (y == 1)); fp = np.sum(pos & (y == 0))
        out.append(vp / n - fp / n * t / (1 - t))
    return np.array(out)

RANGOS = {'sofa_respuesta': (0.40, 0.95), 'hemodinamico': (0.10, 0.90), 'progresion': (0.01, 0.30)}
filas_dca = []
fig, ax = plt.subplots(len(familias), 3, figsize=(15, 3.8 * len(familias)), squeeze=False)
for j, (des, (_, titulo)) in enumerate(DESENLACES.items()):
    um = np.linspace(*RANGOS[des], 60)
    for i, fam in enumerate(familias):
        a = ax[i, j]; todos = None
        for cj in CONJUNTOS_H:
            g = pred[(pred.desenlace == des) & (pred.conjunto == cj) & (pred.familia == fam)]
            if g.empty: continue
            if des == 'progresion': y, p = g.y.to_numpy(), g.p_calibrada.to_numpy()
            else: y, p = 1 - g.y.to_numpy(), 1 - g.p_calibrada.to_numpy()
            bn = beneficio_neto(y, p, um)
            a.plot(um, bn, color=COLOR[cj], lw=2, label=cj)
            todos = y.mean() - (1 - y.mean()) * um / (1 - um)
            filas_dca += [{'desenlace': des, 'familia': fam, 'conjunto': cj, 'umbral': t, 'beneficio_neto': b,
                           'intervenir_a_todos': tt} for t, b, tt in zip(um, bn, todos)]
        if todos is not None:
            a.plot(um, todos, color='gray', lw=1, label='Intervenir a todos'); a.axhline(0, color='k', lw=1, label='A nadie')
            a.set_ylim(-0.05, max(0.05, np.nanmax(todos) * 1.15 + 0.02))
        evento = 'progresión' if des == 'progresion' else 'no respuesta'
        a.set_title(f'{titulo} · {fam}\n(evento = {evento})', fontsize=10); a.legend(fontsize=7)
        if i == len(familias) - 1: a.set_xlabel('Umbral de probabilidad')
        if j == 0: a.set_ylabel('Beneficio neto')
plt.tight_layout(); plt.savefig(f'{SALIDA}/curvas_decision.png', dpi=200, bbox_inches='tight'); plt.show()
dca = pd.DataFrame(filas_dca); dca.to_csv(f'{SALIDA}/curvas_decision.csv', index=False)

# Summary: % of thresholds at which the model beats both default strategies
dca['util'] = dca.beneficio_neto > np.maximum(dca.intervenir_a_todos, 0)
display(dca.groupby(['desenlace', 'familia', 'conjunto']).util.mean().mul(100).round(0).unstack('conjunto')
        .rename(columns=lambda c: f'% umbrales útil · {c}'))

## 5. Out-of-fold SHAP: per-fold refit
For each outcome, feature set and family (Elastic Net, XGBoost):
1. Rebuild **the same 5 outer folds** as the pipeline (same filter, same order, seed 42).
2. In each fold, refit the model with **the hyperparameters chosen in that fold** (`registro_configuraciones.csv`) and the same 1/n patient weights.
3. Compute SHAP values for the **test** windows of that fold (patients never seen by the model).
4. **Reproducibility check**: refit predictions must match the stored `p_sin_calibrar` (correlation ~ 1).

SHAP values are on the log-odds scale (the model's scale, before Platt calibration).

In [ ]:
def pesos_por_paciente(g):
    s = pd.Series(g); return (1.0 / s.map(s.value_counts())).to_numpy()

class FiltroWelch(BaseEstimator, TransformerMixin):  # not used for SHAP; kept for compatibility
    def __init__(self, umbral=0.20): self.umbral = umbral

def base_modelo(fam):
    if fam == 'elastic_net':
        return Pipeline([('imp', SimpleImputer(strategy='median')), ('esc', StandardScaler()),
                         ('model', LogisticRegression(penalty='elasticnet', solver='saga', max_iter=5000, random_state=SEMILLA))])
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('model', XGBClassifier(n_estimators=300, subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                                             eval_metric='logloss', n_jobs=1, random_state=SEMILLA, verbosity=0))])

def _parse(v):
    return None if v in ('None', None) else v

SHAP_FAMILIAS = ['elastic_net', 'xgboost']
shap_filas, control = [], []
for des, (lab, _) in DESENLACES.items():
    d = df[(df['incl_hrv_6h'] == 1) & df[lab].notna()].reset_index(drop=True)
    y = d[lab].astype(int).to_numpy(); g = d['icustay_id'].to_numpy()
    part = list(StratifiedGroupKFold(n_splits=N_EXTERNAS, shuffle=True, random_state=SEMILLA).split(d, y, g))
    for cj in CONJUNTOS_H:
        X = d[CONJ[cj]].to_numpy(dtype=float)
        for fam in SHAP_FAMILIAS:
            r = reg[(reg.desenlace == des) & (reg.conjunto == cj) & (reg.familia == fam)]
            if r.empty: print('sin registro', des, cj, fam); continue
            mejores = json.loads(r.mejores_hiperparametros.iloc[0])
            oof = pred[(pred.desenlace == des) & (pred.conjunto == cj) & (pred.familia == fam)].set_index(['icustay_id', 'hour_index'])
            for k, (tr, te) in enumerate(part):
                m = clone(base_modelo(fam)).set_params(**{a: _parse(b) for a, b in mejores[k].items()})
                m.fit(X[tr], y[tr], model__sample_weight=pesos_por_paciente(g[tr]))
                p_re = m.predict_proba(X[te])[:, 1]
                p_or = oof.loc[list(zip(g[te], d.hour_index.to_numpy()[te])), 'p_sin_calibrar'].to_numpy()
                control.append({'desenlace': des, 'conjunto': cj, 'familia': fam, 'particion': k,
                                'corr': np.corrcoef(p_re, p_or)[0, 1], 'max_dif': np.abs(p_re - p_or).max()})
                Xtr = m[:-1].transform(X[tr]); Xte = m[:-1].transform(X[te])
                if fam == 'elastic_net':
                    ex = shap.LinearExplainer(m.named_steps['model'], Xtr)
                else:
                    ex = shap.TreeExplainer(m.named_steps['model'])
                sv = ex.shap_values(Xte)
                sv = sv[1] if isinstance(sv, list) else sv
                xs = pd.DataFrame(X[te], columns=CONJ[cj])  # original values (for plots)
                sh = pd.DataFrame(sv, columns=CONJ[cj])
                sh['desenlace'], sh['conjunto'], sh['familia'], sh['particion'] = des, cj, fam, k
                sh['icustay_id'], sh['hour_index'] = g[te], d.hour_index.to_numpy()[te]
                xs = xs.add_prefix('x__')
                shap_filas.append(pd.concat([sh.reset_index(drop=True), xs.reset_index(drop=True)], axis=1))
            print(f'{des:15s} {cj:15s} {fam:12s} SHAP listo')

ctrl = pd.DataFrame(control)
print('\nControl de reproducibilidad (reajuste vs OOF guardadas):')
display(ctrl.groupby(['desenlace', 'conjunto', 'familia']).agg(corr_min=('corr', 'min'), max_dif=('max_dif', 'max')).round(4))
SH = pd.concat(shap_filas, ignore_index=True)
SH.to_csv(f'{SALIDA}/shap_oof.csv', index=False)

## 6. SHAP: global and group importance
**Importance** = mean |SHAP| (log-odds). Group importance = **sum** over its variables, also shown as % of the model total.

**Caution:** SHAP shows what the model *uses*, not whether it *improves* prediction. A group with many variables (HRV = 41) accumulates importance even if each adds noise; an overfitted model can give HRV a large share while its AUROC falls. Mean importance per variable is therefore also reported, and results must be read together with the H1 bootstrap.

In [ ]:
imp = []
for (des, cj, fam), s in SH.groupby(['desenlace', 'conjunto', 'familia']):
    for v in CONJ[cj]:
        imp.append({'desenlace': des, 'conjunto': cj, 'familia': fam, 'variable': v, 'grupo': grupo(v),
                    'resumen_hrv': resumen(v), 'media_abs_shap': s[v].abs().mean()})
imp = pd.DataFrame(imp)
imp['pct'] = imp.media_abs_shap / imp.groupby(['desenlace', 'conjunto', 'familia']).media_abs_shap.transform('sum') * 100
imp.to_csv(f'{SALIDA}/shap_importancia_variables.csv', index=False)

# 6a. By group (integrada_sofa): weight of HRV versus clinical variables
pg = imp[imp.conjunto == 'integrada_sofa'].groupby(['desenlace', 'familia', 'grupo']).pct.sum().unstack('grupo').round(1)
pg['HRV total'] = pg[[c for c in pg.columns if c.startswith('HRV')]].sum(axis=1)
print('% de la importancia SHAP por grupo — integrada_sofa'); display(pg)
pg.to_csv(f'{SALIDA}/shap_pct_por_grupo.csv')

pm = imp[imp.conjunto == 'integrada_sofa'].groupby(['desenlace', 'familia', 'grupo']).media_abs_shap.mean().unstack('grupo')
pm = (pm.div(pm.max(axis=1), axis=0) * 100).round(0)
print('Importancia MEDIA POR VARIABLE de cada grupo (100 = el grupo con mayor importancia media en ese modelo)'); display(pm)
pm.to_csv(f'{SALIDA}/shap_media_por_variable_grupo.csv')

# 6b. Within HRV: levels vs variability vs trends
ph = imp[(imp.conjunto == 'integrada_sofa') & imp.resumen_hrv.notna()]
ph = (ph.groupby(['desenlace', 'familia', 'resumen_hrv']).media_abs_shap.sum()
        / ph.groupby(['desenlace', 'familia']).media_abs_shap.sum() * 100).unstack('resumen_hrv').round(1)
print('Dentro de la HRV (% de la importancia HRV) — nivel vs variabilidad vs tendencia'); display(ph)
ph.to_csv(f'{SALIDA}/shap_hrv_nivel_vs_tendencia.csv')

# 6c. Stacked bar chart by group
orden = ['SOFA', 'PAM y lactato (12 h)', 'Clínicas en T0', 'HRV temporal', 'HRV frecuencial', 'HRV no lineal']
colg = dict(zip(orden, ['#4c72b0', '#55a868', '#8c8c8c', '#c44e52', '#dd8452', '#8172b3']))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for a, fam in zip(ax, SHAP_FAMILIAS):
    t = pg.xs(fam, level='familia')[[c for c in orden if c in pg.columns]]
    t.rename(index={k: v[1] for k, v in DESENLACES.items()}).plot.barh(stacked=True, ax=a, color=[colg[c] for c in t.columns], width=.7)
    a.set_title(f'integrada_sofa · {fam}'); a.set_xlabel('% de la importancia SHAP total'); a.set_xlim(0, 100); a.get_legend().remove()
fig.legend(*ax[0].get_legend_handles_labels(), loc='lower center', ncol=6, fontsize=9, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout(); plt.savefig(f'{SALIDA}/shap_por_grupo.png', dpi=200, bbox_inches='tight'); plt.show()

## 7. SHAP: individual variables
Beeswarm (top 15) per outcome for `integrada_sofa`. Each point is a window; colour = variable value (red high, blue low); to the right = pushes towards class 1.

In [ ]:
for fam in SHAP_FAMILIAS:
    for des, (_, titulo) in DESENLACES.items():
        s = SH[(SH.desenlace == des) & (SH.conjunto == 'integrada_sofa') & (SH.familia == fam)]
        if s.empty: continue
        v = CONJ['integrada_sofa']
        expl = shap.Explanation(values=s[v].to_numpy(), data=s[[f'x__{c}' for c in v]].to_numpy(), feature_names=v)
        plt.figure()
        shap.plots.beeswarm(expl, max_display=15, show=False)
        plt.title(f'{titulo} · integrada_sofa · {fam}', fontsize=11)
        plt.savefig(f'{SALIDA}/beeswarm_{des}_{fam}.png', dpi=200, bbox_inches='tight'); plt.show()

print('Top 10 variables por modelo (media |SHAP|):')
top = (imp[imp.conjunto == 'integrada_sofa'].sort_values('media_abs_shap', ascending=False)
       .groupby(['desenlace', 'familia']).head(10)[['desenlace', 'familia', 'variable', 'grupo', 'pct']].round(1))
display(top)

## 8. Dependence plots for key HRV variables
Variable value versus its out-of-fold SHAP value for the long-term and fractal metrics (SDNN, SD2, DFA alpha1, window mean) in XGBoost. Line = LOWESS.

In [ ]:
vars_dep = ['SDNN_mean', 'SD2_mean', 'DFA_a1_mean']
fig, ax = plt.subplots(3, 3, figsize=(14, 11))
for i, (des, (_, titulo)) in enumerate(DESENLACES.items()):
    s = SH[(SH.desenlace == des) & (SH.conjunto == 'integrada_sofa') & (SH.familia == 'xgboost')]
    for j, v in enumerate(vars_dep):
        a = ax[i, j]
        if s.empty: continue
        x, yv = s[f'x__{v}'].to_numpy(), s[v].to_numpy(); ok = ~np.isnan(x)
        a.scatter(x[ok], yv[ok], s=6, alpha=.3, color='#555')
        l = lowess(yv[ok], x[ok], frac=.5); a.plot(l[:, 0], l[:, 1], color='#d62728', lw=2)
        a.axhline(0, color='k', lw=.8); a.set_title(f'{titulo} · {v}', fontsize=10)
        if j == 0: a.set_ylabel('SHAP (log-odds)')
plt.tight_layout(); plt.savefig(f'{SALIDA}/dependencia_hrv.png', dpi=200, bbox_inches='tight'); plt.show()

## 9. Automatic summary and download

In [ ]:
print('=== CALIBRACIÓN (probabilidad calibrada, por desenlace) ===')
for des in DESENLACES:
    t = cal[cal.desenlace == des]
    print(f'\n{des}:')
    for _, r in t.iterrows():
        flag = []
        if abs(r.intercepto) > 0.2: flag.append('calibración global desviada')
        if r.pendiente < 0.8: flag.append('predicciones demasiado extremas')
        if r.pendiente > 1.2: flag.append('predicciones demasiado moderadas')
        if r.brier_escalado <= 0: flag.append('no mejor que la prevalencia')
        print(f'  {r.conjunto:15s} {r.familia:14s} int {r.intercepto:+.2f} ({r.intercepto_ic}) | pend {r.pendiente:.2f} ({r.pendiente_ic}) '
              f'| Brier esc. {r.brier_escalado:.3f} | {"; ".join(flag) or "OK"}')

print('\n=== SHAP: % de la importancia que corresponde a HRV en integrada_sofa ===')
display(pg[['HRV total']])

import shutil
from google.colab import files
zipf = shutil.make_archive('/content/calibracion_shap_v5', 'zip', SALIDA)
print(sorted(os.listdir(SALIDA))); files.download(zipf)